# 02 — Wavelet Hash

**Project:** Wavelet Image Similarity  
**Phần phụ trách:** TV3 — Wavelet Hash

## Mục tiêu

Notebook này minh họa quá trình tạo **Wavelet Hash** từ ảnh:

```text
Ảnh đầu vào
    ↓
Grayscale
    ↓
Resize
    ↓
Discrete Wavelet Transform (DWT)
    ↓
Chọn hệ số LL
    ↓
Tính giá trị tham chiếu
    ↓
Binarization
    ↓
Wavelet Hash
```

Notebook được dùng cho mục đích **thử nghiệm và minh họa**. Code production chính của project vẫn nên nằm trong `src/wavelet/wavelet_hash.py`.

## 1. Import thư viện

Các thư viện chính:

- `numpy`: xử lý mảng ảnh và vector hash.
- `matplotlib`: hiển thị ảnh và kết quả.
- `Pillow`: đọc và tiền xử lý ảnh.
- `PyWavelets`: thực hiện Discrete Wavelet Transform.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pywt

from PIL import Image

## 2. Thiết lập đường dẫn project

Notebook nằm trong:

```text
Wavelet_Image_Similarity/
└── notebooks/
    └── 02_wavelet_hash.ipynb
```

Vì vậy khi chạy notebook, thư mục gốc project được lấy là thư mục cha của `notebooks/`.

Nếu notebook được chạy từ môi trường khác, có thể thay `PROJECT_ROOT` bằng đường dẫn tuyệt đối của project.

In [ ]:
# Xác định thư mục gốc của project.
PROJECT_ROOT = Path.cwd().resolve()

# Nếu đang chạy trực tiếp từ thư mục notebooks,
# chuyển lên thư mục cha.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)

## 3. Hàm đọc và tiền xử lý ảnh

Wavelet Hash cần đưa các ảnh về cùng một kích thước và cùng một không gian màu trước khi biến đổi.

Các bước:

1. Mở ảnh.
2. Chuyển sang grayscale.
3. Resize về kích thước cố định.
4. Chuyển thành `numpy.ndarray`.

In [ ]:
def load_and_preprocess(image_path, size=(64, 64)):
    """Đọc ảnh, chuyển grayscale và resize về kích thước cố định."""
    image = Image.open(image_path).convert("L")
    image = image.resize(size, Image.Resampling.LANCZOS)

    # Chuyển ảnh sang mảng NumPy dạng float.
    array = np.asarray(image, dtype=np.float32)

    return array

## 4. Kiểm tra một ảnh mẫu

Cell dưới đây tự tìm một ảnh trong dataset.

Nếu project chưa có ảnh trong các thư mục dataset, cell sẽ thông báo để người dùng bổ sung dữ liệu thay vì tự tạo số liệu giả.

In [ ]:
candidate_dirs = [
    PROJECT_ROOT / "data" / "input" / "similar",
    PROJECT_ROOT / "data" / "input" / "dissimilar",
]

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

sample_image_path = None

for directory in candidate_dirs:
    if directory.exists():
        for path in sorted(directory.rglob("*")):
            if path.suffix.lower() in image_extensions:
                sample_image_path = path
                break
    if sample_image_path is not None:
        break

print("Ảnh mẫu:", sample_image_path)

In [ ]:
if sample_image_path is not None:
    sample_image = load_and_preprocess(sample_image_path)

    plt.figure(figsize=(5, 5))
    plt.imshow(sample_image, cmap="gray")
    plt.title(f"Preprocessed image: {sample_image_path.name}")
    plt.axis("off")
    plt.show()
else:
    print("Chưa tìm thấy ảnh mẫu trong data/input/.")

## 5. Discrete Wavelet Transform (DWT)

DWT phân rã ảnh thành bốn thành phần ở một mức:

- `LL`: thành phần tần số thấp, chứa thông tin tổng quát của ảnh.
- `LH`: thông tin chi tiết theo một hướng.
- `HL`: thông tin chi tiết theo hướng khác.
- `HH`: chi tiết tần số cao.

Wavelet Hash tập trung vào phần `LL` vì đây là thành phần chứa cấu trúc tổng quát của ảnh.

In [ ]:
def dwt_decompose(image, wavelet="haar"):
    """Thực hiện DWT một mức và trả về LL, LH, HL, HH."""
    ll, (lh, hl, hh) = pywt.dwt2(image, wavelet)

    return ll, lh, hl, hh

In [ ]:
if sample_image_path is not None:
    ll, lh, hl, hh = dwt_decompose(sample_image)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    components = [
        ("LL", ll),
        ("LH", lh),
        ("HL", hl),
        ("HH", hh),
    ]

    for ax, (name, component) in zip(axes, components):
        ax.imshow(component, cmap="gray")
        ax.set_title(name)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 6. Tạo Wavelet Hash

Quy trình trong notebook:

1. Resize ảnh về `64 × 64`.
2. Thực hiện DWT một mức.
3. Lấy ma trận `LL`.
4. Tính giá trị trung bình của `LL`.
5. So sánh từng hệ số `LL` với giá trị trung bình.
6. Giá trị lớn hơn hoặc bằng trung bình → `1`.
7. Giá trị nhỏ hơn trung bình → `0`.

Kết quả cuối cùng là một chuỗi bit.

> **Lưu ý:** Đây là cách triển khai Wavelet Hash chuẩn dùng cho notebook minh họa. Khi `src/wavelet/wavelet_hash.py` của nhóm có quy ước riêng về kích thước, wavelet hoặc cách chọn ngưỡng, cần thống nhất notebook với implementation chính thức của project.

In [ ]:
def wavelet_hash(
    image,
    wavelet="haar",
    hash_size=(8, 8),
):
    """
    Tạo Wavelet Hash từ ảnh grayscale đã được resize.

    Parameters
    ----------
    image : numpy.ndarray
        Ảnh grayscale.

    wavelet : str
        Tên wavelet dùng cho DWT.

    hash_size : tuple
        Kích thước hash sau khi lấy LL.

    Returns
    -------
    str
        Chuỗi bit Wavelet Hash.
    """

    # DWT một mức.
    ll, _, _, _ = pywt.dwt2(image, wavelet)

    # Resize LL về kích thước hash mong muốn.
    ll_image = Image.fromarray(ll.astype(np.float32))
    ll_image = ll_image.resize(hash_size, Image.Resampling.BILINEAR)
    ll = np.asarray(ll_image, dtype=np.float32)

    # Giá trị tham chiếu.
    reference = np.mean(ll)

    # Binarization.
    bits = (ll >= reference).astype(np.uint8)

    # Chuyển ma trận bit thành chuỗi.
    return "".join(str(int(bit)) for bit in bits.flatten())

## 7. Tạo hash cho ảnh mẫu

Cell này hiển thị:

- Wavelet Hash.
- Độ dài hash.
- Giá trị hash.

In [ ]:
if sample_image_path is not None:
    sample_hash = wavelet_hash(sample_image)

    print("Ảnh:", sample_image_path)
    print("Wavelet Hash:")
    print(sample_hash)
    print("Độ dài hash:", len(sample_hash))
else:
    print("Chưa có ảnh mẫu để tạo hash.")

## 8. Trực quan hóa ma trận bit của hash

Chuỗi hash có thể chuyển ngược thành ma trận `8 × 8` để quan sát.

In [ ]:
def hash_to_matrix(hash_string, size=(8, 8)):
    """Chuyển chuỗi hash thành ma trận bit."""
    expected_length = size[0] * size[1]

    if len(hash_string) != expected_length:
        raise ValueError(
            f"Hash phải có {expected_length} bit, "
            f"nhưng nhận được {len(hash_string)} bit."
        )

    return np.array(
        [int(bit) for bit in hash_string],
        dtype=np.uint8
    ).reshape(size)

In [ ]:
if sample_image_path is not None:
    hash_matrix = hash_to_matrix(sample_hash)

    plt.figure(figsize=(5, 5))
    plt.imshow(hash_matrix, cmap="gray", interpolation="nearest")
    plt.title("Wavelet Hash")
    plt.xticks(range(8))
    plt.yticks(range(8))
    plt.grid(True)
    plt.show()

## 9. So sánh hai Wavelet Hash

Wavelet Hash không dùng để so sánh trực tiếp bằng khoảng cách Euclid. Hai hash được so sánh bằng **Hamming Distance**.

Hamming Distance là số vị trí bit khác nhau giữa hai chuỗi có cùng độ dài.

Ví dụ:

```text
H1 = 10110110
H2 = 10000110

      ^  ^
```

Khoảng cách bằng số vị trí khác nhau.

In [ ]:
def hamming_distance(hash_a, hash_b):
    """Tính Hamming Distance giữa hai hash."""
    if len(hash_a) != len(hash_b):
        raise ValueError("Hai hash phải có cùng độ dài.")

    return sum(bit_a != bit_b for bit_a, bit_b in zip(hash_a, hash_b))

In [ ]:
# Ví dụ minh họa.
hash_a = "10110110"
hash_b = "10000110"

print("Hash A:", hash_a)
print("Hash B:", hash_b)
print("Hamming Distance:", hamming_distance(hash_a, hash_b))

## 10. Tạo hash cho một cặp ảnh

Notebook tự tìm cặp ảnh đầu tiên trong dataset `similar/pair_*`.

Nếu dataset chưa có đúng cấu trúc này, cell sẽ thông báo thay vì tự tạo dữ liệu.

In [ ]:
similar_root = PROJECT_ROOT / "data" / "input" / "similar"

pair_paths = []

if similar_root.exists():
    for pair_dir in sorted(similar_root.iterdir()):
        if pair_dir.is_dir():
            images = [
                p for p in sorted(pair_dir.iterdir())
                if p.suffix.lower() in image_extensions
            ]

            if len(images) >= 2:
                pair_paths = images[:2]
                break

if len(pair_paths) == 2:
    image_a = load_and_preprocess(pair_paths[0])
    image_b = load_and_preprocess(pair_paths[1])

    hash_a = wavelet_hash(image_a)
    hash_b = wavelet_hash(image_b)

    distance = hamming_distance(hash_a, hash_b)

    print("Ảnh 1:", pair_paths[0])
    print("Ảnh 2:", pair_paths[1])
    print()
    print("Hash 1:", hash_a)
    print("Hash 2:", hash_b)
    print("Hamming Distance:", distance)
else:
    print("Chưa tìm thấy một cặp ảnh hợp lệ trong data/input/similar/.")

## 11. Thử nghiệm với nhiều wavelet

Wavelet Hash có thể được thử với nhiều loại wavelet.

Ví dụ:

- `haar`
- `db1`
- `db2`

Kết quả thực tế phụ thuộc vào dataset và implementation của project. Notebook chỉ tạo dữ liệu thử nghiệm khi ảnh thực tế tồn tại.

In [ ]:
if sample_image_path is not None:
    wavelet_names = ["haar", "db1", "db2"]

    print("So sánh Wavelet Hash trên ảnh mẫu:")
    print("-" * 70)

    for wavelet_name in wavelet_names:
        current_hash = wavelet_hash(
            sample_image,
            wavelet=wavelet_name
        )

        print(
            f"{wavelet_name:>6} | "
            f"length = {len(current_hash):2d} | "
            f"hash = {current_hash}"
        )
else:
    print("Chưa có ảnh mẫu để thử nghiệm.")

## 12. Kiểm tra tính ổn định với hai ảnh trong cùng một cặp

Mục đích là xem hai ảnh được xem là `similar` trong dataset tạo ra Wavelet Hash có khoảng cách Hamming như thế nào.

Kết quả này có thể được dùng làm đầu vào cho phần **Similarity / Evaluation** của project.

In [ ]:
if len(pair_paths) == 2:
    print("Pair:", pair_paths[0].parent.name)
    print("Hamming Distance:", distance)

    print()
    print("Hash 1:", hash_a)
    print("Hash 2:", hash_b)
else:
    print("Không có cặp ảnh để kiểm tra.")

## 13. Pipeline hoàn chỉnh

```text
Image
  │
  ▼
Convert to Grayscale
  │
  ▼
Resize 64 × 64
  │
  ▼
DWT
  │
  ▼
LL Sub-band
  │
  ▼
Resize LL → 8 × 8
  │
  ▼
Mean of LL
  │
  ▼
Binary Threshold
  │
  ▼
Wavelet Hash
  │
  ▼
Hamming Distance
  │
  ▼
Similarity Decision
```

Notebook này tập trung vào **TV3 — Wavelet Hash**. Việc quyết định ảnh `Similar/Dissimilar` theo threshold và đánh giá Accuracy/ROC thuộc phần Evaluation của project.

## 14. Kết luận

Notebook đã minh họa:

- Đọc và tiền xử lý ảnh.
- Grayscale và resize.
- Discrete Wavelet Transform.
- Phân rã ảnh thành `LL`, `LH`, `HL`, `HH`.
- Sử dụng `LL` để tạo Wavelet Hash.
- Binarization bằng giá trị trung bình.
- Chuyển hash thành chuỗi bit.
- Tính Hamming Distance.
- Thử nghiệm nhiều loại wavelet.
- Áp dụng pipeline trên dữ liệu thực tế nếu dataset của project tồn tại.

**Không điền Accuracy, AUC hoặc các số liệu đánh giá giả.** Các số liệu đó phải được lấy từ kết quả chạy thực tế của dataset.